In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor

from sklearn.feature_extraction.text import TfidfVectorizer
import nltk

In [ ]:
nltk.download('words', download_dir='.\\.venv\\Lib\\nltk_data')
nltk.download('punkt_tab', download_dir='.\\.venv\\Lib\\nltk_data')

nltk.data.path.append('.\\.venv\\Lib\\nltk_data')

In [ ]:
final_df = pd.read_csv('./data/finalCards.csv')

In [ ]:
creature_dummies_before = [
    'isAlternative',
    'isGameChanger', 
    'isPromo',
    'isReprint', 
    'isReserved',
    'isHuman', 
    'isElemental',
    'isDragon', 
    'isSpirit', 
    'isAngel', 
    'isElf', 
    'isVampire', 
    'isZombie',
    'isBeast', 
    'isWizard', 
    'isSoldier', 
    'isKnight', 
    'isCleric', 
    'isWarrior',
    'isRogue', 
    'isShaman', 
    'isDruid', 
    'isCreature', 
    'isPlaneswalker', 
    'isMTGO', 
    'isFlying',
    'isLegal',
    'isBanned']

for col in creature_dummies_before:
    final_df[col] = final_df[col].map({True: 1, False: 0})

In [ ]:
creatures_df = final_df[final_df['isCreature'] == 1]
creatures_df = creatures_df[['uuid', 
                            'cardName', 
                            'availability', 
                            'colorIdentity', 
                            'edhrecRank', 
                            'edhrecSaltiness', 
                            'finishes', 
                            'isAlternative', 
                            'isGameChanger',
                            'isPromo',
                            'isReprint',
                            'isReserved',
                            'manaValue',
                            'cardNumber',
                            'power',
                            'rarity',
                            'setCode',
                            'subtypes',
                            'text',
                            'toughness',
                            'types',
                            'price',
                            'commander',
                            'setName',
                            'releaseDate',
                            'scryfallId',
                            'isHuman', 
                            'isElemental',
                            'isDragon', 
                            'isSpirit', 
                            'isAngel', 
                            'isElf',
                            'isVampire', 
                            'isZombie',
                            'isBeast', 
                            'isWizard', 
                            'isSoldier', 
                            'isKnight', 
                            'isCleric', 
                            'isWarrior',
                            'isRogue', 
                            'isShaman', 
                            'isDruid', 
                            'isCreature', 
                            'isMTGO', 
                            'isFlying',
                            'isLegal',
                            'isBanned'
                            ]]

print(len(creatures_df))
creatures_df = creatures_df.dropna()
print(len(creatures_df))

In [ ]:
creatures_df['text'] = creatures_df['text'].str.replace(r'\\n', ' ', regex=True)

In [ ]:
X = creatures_df[
    [
        'power', 
        'toughness', 
        'price', 
        'edhrecRank', 
        'manaValue', 
        'isHuman', 
        'isElemental',
        'isDragon', 
        'isSpirit', 
        'isAngel', 
        'isElf',
        'isVampire', 
        'isZombie',
        'isBeast', 
        'isWizard', 
        'isSoldier', 
        'isKnight', 
        'isCleric', 
        'isWarrior',
        'isRogue', 
        'isShaman', 
        'isDruid', 
        'isCreature', 
        'isMTGO', 
        'isFlying',
        'isLegal',
        'isBanned',
        'text' # remove later
        ]]

y = creatures_df['edhrecSaltiness']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
MIN_DF = 100 # might be too strict a cutoff?
MAX_DF = 500 # remove the word "creature"/"creatures"

vectorizer = TfidfVectorizer(min_df=MIN_DF, max_df=MAX_DF, stop_words='english')
vectorizer.fit(X_train.text)
X_train_text = vectorizer.fit_transform(X_train.text)

feature_names = vectorizer.get_feature_names_out()

print('size:' ,X_train_text.shape)
print('vocab:' ,feature_names)

In [ ]:
X_test_text = vectorizer.transform(X_test.text)
print('size:' ,X_test_text.shape)

In [ ]:
clf = KNeighborsRegressor(n_neighbors=1)
clf.fit(X_train_text, y_train)
y_pred_text = clf.predict(X_test_text)

In [ ]:
mae_text = mean_absolute_error(y_test, y_pred_text)
mse_text = mean_squared_error(y_test, y_pred_text)
r2_text = r2_score(y_test, y_pred_text)

print(f'Mean Absolute Error: {mae_text}')
print(f'Mean Squared Error: {mse_text}')
print(f'R-squared: {r2_text}')